In [ ]:
import numpy as np
import pandas as pd

In [ ]:
go_df = pd.read_csv('./EnrichmentResults/Regulatory Variants Results/HighVariants_Prioritized_GO_Enrichment.csv')
kegg_df = pd.read_csv('./EnrichmentResults/Regulatory Variants Results/HighVariants_Prioritized_GO_Enrichment.csv')
reactome_df = pd.read_csv('./EnrichmentResults/Regulatory Variants Results/HighVariants_Prioritized_Reactome.csv')
variants_df = pd.read_csv('./PrioritizedVariants_Regulatory_High.csv')

In [ ]:
def merge_enrichment_results(variant_df, infectivity_class, gene_ontology_df=None, kegg_df=None, reactome_pathways_df=None):
    # helper function to safely extract genes from a dataframe if it exists
    def extract_genes(df):
        if df is not None and not df.empty and 'geneID' in df.columns:
            # drop NaN values and split/explode
            return set(df['geneID'].dropna().str.split('/').explode())
        return set()

    # extract genes for each tool
    go_genes = extract_genes(gene_ontology_df)
    kegg_genes = extract_genes(kegg_df)
    reactome_genes = extract_genes(reactome_pathways_df)
    
    # combine all found genes (if all sets are empty, union will just be an empty set)
    all_enriched_genes = set().union(go_genes, kegg_genes, reactome_genes)
    
    # if no genes were found in any provided enrichment table, return an empty dataframe matching schema
    if not all_enriched_genes:
        print("Warning: No enrichment data provided or all provided tables were empty.")
        # return an empty dataframe with the expected columns
        empty_cols = ['gene_symbol', 'consensus_score', 'supporting_tools'] + list(variant_df.columns)
        return pd.DataFrame(columns=empty_cols)

    gene_scores = []

    for gene in all_enriched_genes:
        score = 0
        tools = []
        if gene in go_genes:
            score += 1
            tools.append('GO_CC')
        if gene in kegg_genes:
            score += 1
            tools.append('KEGG')
        if gene in reactome_genes:
            score += 1
            tools.append('Reactome')

        gene_scores.append({
            'gene_symbol': gene,
            'consensus_score': score,
            'supporting_tools': ", ".join(tools)
        })

    score_df = pd.DataFrame(gene_scores)
    
    # merge with the variants dataframe 
    master_table = pd.merge(score_df, variant_df, left_on='gene_symbol', right_on='SYMBOL', how='inner')
    
    master_table.to_csv(f'./Regulatory_PlausibleGenes_fromEnrichmentAnalysis_{infectivity_class}.csv', index=False)

    return master_table

In [ ]:
# example usage for Regulatory Variants of the low-infectivity class where no reactome pathways exist 

master_table = merge_enrichment_results(variants_df, infectivity_class = "Low", gene_ontology_df=go_df, kegg_df=kegg_df, reactome_pathways_df=None)
master_table